# Heat Diffusion — Google Colab edition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/otmanon/cs_academy_heat_diffusion/blob/tree/main/notebooks)

Pour a cup of hot coffee and it slowly cools; heat always spreads from hot
places to cold ones until everything evens out. In this notebook you'll
teach a computer to **simulate** that spreading and watch a glowing hot spot
melt away across a virtual frying pan.

This is the **Colab edition** — it runs entirely in your browser with nothing
to install and needs no project files. Your one job is to fill in the
`update_temperature` function in section 4; everything else is ready to run.

## 1. Setup

Colab already ships with NumPy and Matplotlib, so there's nothing to install —
the cell below just imports them. (The `pip install` line is a harmless
safety net in case you're on a stripped-down environment.)

In [ ]:
# Colab already includes these; the install is a no-op safety net.
!pip install -q numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Draw plots and animations inline in the notebook.
%matplotlib inline

## 2. The Temperature Grid

We store our temperatures on a square grid, so that each point in an (X, Y)
grid has a different temperature. (We use **kelvin** so we can act more like
scientists — 373 K is boiling, room temperature is about 293 K.)

See **Lesson 0: The Temperature Grid** for the full story.

In [ ]:
# Define grid parameters
grid_size = 10  # number of points in the x, y directions
initial_temperature = np.zeros((grid_size, grid_size))  # initialize grid with zeros

# Set initial temperature values
initial_temperature[4:6, 4:6] = 373  # a square region in the center is boiling hot!

Which we can plot with matplotlib:

In [ ]:
# Visualize initial temperature
fig = plt.figure(figsize=(8, 6))
img = plt.imshow(initial_temperature, cmap='hot')
cbar = plt.colorbar(label='Temperature (K)')
cbar.set_label('Temperature (K)', rotation=270, labelpad=15)
plt.title('Initial Temperature Distribution')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, which='both', linestyle='--', linewidth=0.5, color='gray')
plt.minorticks_on()
plt.show()

You can see the temperature visualized on the pixels of this image, where
lighter colors are hotter. This is kind of like the temperature distribution
on a frying pan with a hot spot in the middle.

## 3. Physical Parameters

We're working with temperature in kelvin, but what other physical units are
relevant here?

### 1. Width of the grid
How wide / tall is the entire grid? From this and `grid_size` we get `dx`, the
spacing between two neighboring grid points.

### 2. Timestep
We break time into small **timesteps** and update the physics each step. A
timestep of 0.01 s means 100 temperature updates every second.

### 3. [Thermal Diffusivity](https://en.wikipedia.org/wiki/Thermal_diffusivity)
Thermal diffusivity, denoted $\alpha$, is the rate at which heat transfers
inside a material — a dial for how eagerly heat spreads.

> **Stability (Lesson 3):** an explicit scheme only stays stable while
> $\alpha\,dt/dx^2 \le 1/4$. The cell below prints this number — if you crank
> up `alpha` or `timestep` and the simulation explodes into noise, this is why.

In [ ]:
width = 1                    # total grid width in meters
dx = width / grid_size       # spacing between adjacent grid points (meters)
timestep = 0.01              # 100 timesteps per second (seconds)
alpha = 0.1                  # thermal diffusivity (m^2 / s)

stability = alpha * timestep / dx**2
print('stability number =', stability, '(must be <= 0.25 to stay stable)')

## 4. The Update Rule  — *your task*

Now for the heart of the simulator. The **heat equation** says every point
drifts toward the average of its four neighbors. Made into arithmetic (the
**5-point stencil**, see Lesson 2), the new temperature at each interior point
is:

$$
T^{\text{new}}_{i,j} = T_{i,j} + \alpha\,dt\;\frac{T_{i+1,j} + T_{i-1,j} + T_{i,j+1} + T_{i,j-1} - 4\,T_{i,j}}{dx^2}
$$

**Your task:** fill in `update_temperature` so it applies this formula to every
interior point of the grid. Try to do it with NumPy slicing (no `for` loops) —
Lesson 2 walks through exactly how the neighbor slices line up.

In [ ]:
# Compute the temperature after a single timestep.
# T is the current temperature grid; return the updated grid Tnew.
def update_temperature(T, alpha, timestep, dx):
    Tnew = T.copy()

    # TODO: replace the line above's copy with the 5-point stencil update.
    # Hint: the four neighbor grids are just T shifted by one row/column:
    #   below = T[2:,   1:-1]
    #   above = T[:-2,  1:-1]
    #   right = T[1:-1, 2:]
    #   left  = T[1:-1, :-2]
    #   center= T[1:-1, 1:-1]
    # laplacian = (below + above + right + left - 4 * center) / dx**2
    # Tnew[1:-1, 1:-1] = center + alpha * timestep * laplacian

    return Tnew

## 5. Animating the Simulation

The `animate` function below calls `update_temperature` once per frame. After
each update it applies **insulated (Neumann) boundary conditions** — each edge
copies its inner neighbor so no heat escapes the pan (see Lesson 3).

In [ ]:
# Update the plot for each time step
def animate(frame):
    global T
    T = update_temperature(T, alpha, timestep, dx)

    # apply insulated boundary conditions (no heat escapes the edges)
    T[0, :] = T[1, :]
    T[-1, :] = T[-2, :]
    T[:, 0] = T[:, 1]
    T[:, -1] = T[:, -2]

    img.set_array(T)
    return img,

Run the cell below to build and display the animation. If `update_temperature`
is still the empty stub, the hot spot will just sit there — come back and run
it again once you've filled the function in.

In [ ]:
T = initial_temperature.copy()  # start from the initial hot spot
seconds = 1
num_frames = int(seconds / timestep)
ani = FuncAnimation(fig, animate, frames=num_frames, interval=20, repeat=False)

from IPython.display import HTML
HTML(ani.to_jshtml())

## 6. Experiment

Once it's working, try changing things and re-running from section 2:

- **Different diffusivity:** raise or lower `alpha`. Watch the hot spot spread
  faster or slower.
- **Different initial heat:** edit `initial_temperature` — try two hot spots,
  a hot stripe, or a cold spot in a warm field.
- **Break stability:** slowly increase `timestep` until the stability number
  passes 0.25 and watch the simulation explode into noise (Lesson 3).
- **Fixed edges:** replace the insulated boundary lines in `animate` with a
  fixed value (e.g. `T[0, :] = 273`) to hold an edge at a set temperature.